## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger
from datetime import datetime
from utils.pipeline_monitoring import log_pipeline_run

dataset_name = "private_vehicles_fact"
layer = "gold"

logger = get_project_logger("Gold_fact_private_vehicles")
logger.info("--- Starting Gold Layer: Creating fact_private_vehicles ---")

## Creating fact view

In [0]:
start_time = datetime.now()
try:
    logger.info("Step 1/4: Running SQL to create fact_private_vehicles view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_private_vehicles AS
    SELECT
    p.car_num,
    m.manufacturer_key,
    f.fuel_type_key,
    c.color_key,
    o.ownership_key,
    p.commercial_nm,
    p.prod_year,
    p.road_entry_dt,
    p.last_test_dt,
    p.test_valid_until_dt,
    1 AS vehicle_count
    
    FROM transport_lakehouse.silver.silver_vehicles_private p
    LEFT JOIN transport_lakehouse.gold.dim_manufacturer m
    ON CAST(p.manufacturer_cd AS STRING) = CAST(m.manufacturer_cd AS STRING)
    LEFT JOIN transport_lakehouse.gold.dim_fuel_type f
    ON p.fuel_type_nm = f.fuel_type_nm
    LEFT JOIN transport_lakehouse.gold.dim_color c
    ON CAST(p.color_cd AS STRING) = CAST(c.color_cd AS STRING)
    LEFT JOIN transport_lakehouse.gold.dim_ownership o
    ON p.ownership = o.ownership
    """
    
    spark.sql(sql_query)
    logger.info("Step 1/4: View created successfully.")
    logger.info("--Performing Quality Check (Row Count)--")
    fact_count = spark.table("transport_lakehouse.gold.fact_private_vehicles").count()
    logger.info(f"Quality Check Passed: fact_private_vehicles contains {fact_count} rows.")
    end_time = datetime.now()

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="SUCCESS",
        rows_ingested=fact_count,
        error_message=None
    )

except Exception as e:
    end_time = datetime.now()
    logger.error(f"Failed to create Gold Fact: {str(e)}")

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="FAILED",
        rows_ingested=0,
        error_message=str(e)
    )

    raise
logger.info("--- Gold fact_private_vehicles Process Completed ---")


## Missing keys check

In [0]:
logger.info("Step 2/4: Starting missing keys check...")

df_check = spark.sql("""
SELECT
    SUM(CASE WHEN manufacturer_key IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer,
    SUM(CASE WHEN color_key IS NULL THEN 1 ELSE 0 END) AS missing_color,
    SUM(CASE WHEN fuel_type_key IS NULL THEN 1 ELSE 0 END) AS missing_fuel_type,
    SUM(CASE WHEN ownership_key IS NULL THEN 1 ELSE 0 END) AS missing_ownership
FROM transport_lakehouse.gold.fact_private_vehicles
""")

results = df_check.collect()[0]

missing_report = (
    f"Missing Keys Summary: "
    f"Manufacturers: {results['missing_manufacturer']}, "
    f"Colors: {results['missing_color']}, "
    f"Fuel: {results['missing_fuel_type']}, "
    f"Ownership: {results['missing_ownership']}"
)

if (results['missing_manufacturer'] + results['missing_color'] + 
    results['missing_fuel_type'] + results['missing_ownership']) > 0:
    logger.warning(f"Step 2/4: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 2/4: Finished missing keys check successfully. All keys are mapped. {missing_report}")

## Duplication check

In [0]:
logger.info("Step 3/4: Starting Duplication check...")

df_check = spark.sql("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT car_num) AS distinct_cars
FROM transport_lakehouse.gold.fact_private_vehicles
""")

results = df_check.collect()[0]

if results['rows'] == results['distinct_cars']:
    logger.info(f"Step 2/4: Finished duplication check successfully. All rows are unique.")
else:
    logger.warning(f"Step 2/4: Finished with DUPLICATE ROWS!")


## Key null ratio check

In [0]:
logger.info("Step 4/4: Starting Key null ratio check...")

df_check = spark.sql("""
SELECT
  CAST(AVG(CASE WHEN manufacturer_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS manufacturer_key_fill_rate,
  CAST(AVG(CASE WHEN ownership_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS ownership_key_fill_rate,
  CAST(AVG(CASE WHEN fuel_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS fuel_type_key_fill_rate,
  CAST(AVG(CASE WHEN color_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS color_key_fill_rate
FROM transport_lakehouse.gold.fact_private_vehicles
""")

results = df_check.collect()[0]

missing_report = (
    f"Missing Keys Summary: "
    f"Manufacturers: {results['manufacturer_key_fill_rate']}, "
    f"Colors: {results['color_key_fill_rate']}, "
    f"Fuel: {results['fuel_type_key_fill_rate']}, "
    f"Ownership: {results['ownership_key_fill_rate']}"
)

if (results['manufacturer_key_fill_rate'] < 1 or 
    results['color_key_fill_rate'] < 1 or 
    results['fuel_type_key_fill_rate'] < 1 or 
    results['ownership_key_fill_rate'] < 1 ):
    logger.warning(f"Step 4/4: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 4/4: Finished  Key null ratio successfully. All keys are mapped. {missing_report}")

